# DLRM (using TorchRec + TorchDistributor + StreamingDataset)
This notebook illustrates how to create a distributed DLRM recommendation model for predicting click-through rates. This notebook was tested on n1-standard-32 with 4*T4 instances on the Colab Enterprise. It uses some code from the Facebook DLRM implementation (which has an MIT License) For more insight into the DLRM recommendation model, see the following resources:

* Facebook Research's repository: https://
github.com/facebookresearch/dlrm
* Nvidia's repository: https://github.com/NVIDIA/DeepLearningExamples/blob/master/PyTorch/Recommendation/DLRM/README.md
* Video overview: https://www.youtube.com/watch?v=r9J3UZmddC4  

Note: Where you see # TODO in this notebook, you may need to enter custom code to ensure that the notebook runs successfully.

# Preparation.  
 Prepare a serverless spark runtime for data/feature processing.

In [ ]:
PROJECT_ID = "project-kangwe-poc"  # @param {type:"string"}
REGION = "us-central1"  # @param {type: "string"}
BUCKET_URI = "gs://project-kangwe-poc-experiments-staging-bucket"  # @param {type:"string"}
EXPERIMENT_NAME = "experiments-tutorial"  # @param {type:"string"}

In [ ]:
! pip3 install --upgrade --quiet google-cloud-aiplatform
! pip3 install --upgrade --quiet google-cloud-aiplatform[autologging]

! gcloud config set project {PROJECT_ID}

# Only if your bucket doesn't already exist: Uncomment the following code to create your Cloud Storage bucket.
# ! gsutil mb -l {REGION} -p {PROJECT_ID} {BUCKET_URI}

In [ ]:
from google.cloud.dataproc_spark_connect import DataprocSparkSession
from google.cloud.dataproc_v1 import Session

import pyspark.sql.functions as f

session = Session()
# Config the session. TODO: add the sample to custom a environment with GPU
# session.runtime_config.properties=
# session.environment_config

# Create the Spark session.
spark = (
   DataprocSparkSession.builder
     .appName("DemoDLRM")
     .dataprocSessionConfig(session)
     .getOrCreate()
)

# Step 1. Saving Data in BigQuery/GCS Storage in the MDS (Mosaic Data Shard) format
This notebook creates a synthetic dataset with 100k rows that will be used to train a DLRM model.

In [ ]:
%pip install -q mosaicml-streaming==0.7.5

# 1.1. Creating a Synthetic Dataset
This notebook creates a synthetic dataset for predicting a binary label given both dense (numerical) and sparse (categorical) data. This synthetic dataset has a similar layout to other publicly available datasets, such as the Criteo click logs dataset. You can update this notebook to support those datasets as long as the data preprocessing is done correctly.

For a tangible example in retail, the numerical columns could represent features like the user's age, product's weight, or time of day, and the sparse columns could represent features like user's location, product's category, and so on. The label column describes the interaction between the user and the product. For example, a positive label of 1 might indicate that the user would buy the product, while a negative label of 0 might indicate that the user would give the product a 1-star rating.

In [ ]:
import pandas as pd
import numpy as np
import random

# Set random seed for reproducibility
random.seed(42)
np.random.seed(42)

# Parameters for dataset generation
num_samples = 100000
num_int_features = 10
num_cat_features = 10

# Define the ranges for each integer feature
int_feature_ranges = [(0, random.randint(501, 10000)) for _ in range(num_int_features)]

# Generate unique category sizes for each categorical feature
cat_feature_categories = [np.random.randint(1, 201) for _ in range(num_cat_features)]

# Generate label column (0 or 1 depending on whether the interaction is negative or positive)
labels = np.random.randint(0, 2, size=num_samples)

# Generate integer features with the specified range
int_features = np.column_stack([
    np.random.randint(low, high, size=num_samples)
    for (low, high) in int_feature_ranges
])

# Generate categorical features with different unique categories
cat_features = np.column_stack([
    np.random.randint(0, num_categories, size=num_samples)
    for num_categories in cat_feature_categories
])

# Combine all features into a DataFrame
data = np.column_stack((labels, int_features, cat_features))
columns = ['label'] + [f'int_{i}' for i in range(1, num_int_features+1)] + [f'cat_{i}' for i in range(1, num_cat_features+1)]
df = pd.DataFrame(data, columns=columns)

# save and loading Spark DataFrame

In [ ]:
from google.cloud import bigquery

# 1. 确保 BigQuery 数据集存在
bq_client = bigquery.Client(project='project-kangwe-poc')
dataset_id = f"{bq_client.project}.recommender_systems"
dataset = bigquery.Dataset(dataset_id)
dataset.location = "US"  # 根据需要修改区域
bq_client.create_dataset(dataset, exists_ok=True)
print(f"Dataset {dataset_id} is ready.")

# 2. 配置参数
# 注意：请确保此 GCS 桶已在控制台中手动创建
temp_bucket = "project-kangwe-poc-dlrm"

# 将 Pandas DataFrame 转换为 Spark DataFrame
spark_df = spark.createDataFrame(df)

# 3. 使用 BigQuery 专用格式写入
spark_df.write.format("bigquery") \
    .option("table", "recommender_systems.dlrm_sample_dataset") \
    .option("temporaryGcsBucket", temp_bucket) \
    .mode("overwrite") \
    .save()

In [ ]:
# 更新表路径以匹配之前保存的名称
df = spark.table("recommender_systems.dlrm_sample_dataset")
display(df)


# 1.2. Preprocessing the Data
If you are using a dataset other than the provided synthetic dataset, update this cell to for preprocessing and data cleaning as needed. For this synthetic dataset, all that is required is to normalize the dense columns.

Note: You can repartition the dataset as needed to help improve performance for this cell.

In [ ]:
from pyspark.ml.feature import StandardScaler, StringIndexer, VectorAssembler
from pyspark.ml import Pipeline
from pyspark.sql.functions import udf, col
from pyspark.sql.types import DoubleType, IntegerType
from pyspark.ml.linalg import VectorUDT

# UDF to extract the first element of the vector
extract_value = udf(lambda vector: float(vector[0]), DoubleType())

# List of dense features
dense_cols = [c for c in df.columns if 'int' in c]

# Normalize dense columns
stages = []
for column in dense_cols:
    assembler = VectorAssembler(inputCols=[column], outputCol=column + "_vec")
    scaler = StandardScaler(inputCol=column + "_vec", outputCol=column + "_scaled", withStd=True, withMean=True)
    stages += [assembler, scaler]

# Define the pipeline
pipeline = Pipeline(stages=stages)
model = pipeline.fit(df)

# Transform the dataframe
df_transformed = model.transform(df)

# Use the UDF to overwrite the scaled vector columns with the single extracted value
for column in dense_cols:
    df_transformed = df_transformed.withColumn(column, extract_value(col(column + "_scaled")))

# Drop the intermediate columns and update the dataframe with transformed dense columns
for column in dense_cols:
    if column in dense_cols:
        df_transformed = df_transformed.drop(column + "_vec").drop(column + "_scaled")

# Display the transformed dataset
display(df_transformed)

In [ ]:
from pyspark.sql.functions import col

# Identifying dense and categorical columns from the Spark DataFrame 'df'
dense_cols = [c for c in df_transformed.columns if 'int' in c]
cat_cols = [c for c in df_transformed.columns if 'cat' in c]

# Calculating the number of unique values (distinct count) for each categorical column
emb_counts = [df_transformed.select(c).distinct().count() for c in cat_cols]

# Printing the results
print(f"The number of rows in df are {df_transformed.count()}")
print(f"There are {len(dense_cols)} dense columns: {dense_cols}")
print(f"There are {len(cat_cols)} categorical columns, where the max embedding count is {max(emb_counts)}:")

# Creating a dictionary to print the categorical column names alongside their unique counts
emb_count_dict = {cat_col: emb_count for cat_col, emb_count in zip(cat_cols, emb_counts)}
print(emb_count_dict)

In [ ]:
# Splitting the dataframe into train, test, and validation sets
train_df, validation_df, test_df = df_transformed.randomSplit([0.7, 0.2, 0.1], seed=42)

# Showing the count of each split to verify our distribution
print(f"Training Dataset Count: {train_df.count()}")
print(f"Validation Dataset Count: {validation_df.count()}")
print(f"Test Dataset Count: {test_df.count()}")

# 1.3. Saving to MDS Format within GCS Storage
In this step, you convert the data to MDS to allow for efficient train/validation/test splitting and then save it to a GCS Storage.

View the Mosaic Streaming guide here for more details:

General details: https://docs.mosaicml.com/projects/streaming/en/stable/
Main concepts: https://docs.mosaicml.com/projects/streaming/en/stable/getting_started/main_concepts.html#dataset-conversion
dataframeToMDS details: https://docs.mosaicml.com/projects/streaming/en/stable/preparing_datasets/spark_dataframe_to_mds.html

In [ ]:
# Add dependency into spark runtime to ensure executors have the library
spark.addArtifacts("mosaicml-streaming==0.7.5", pypi=True)

In [ ]:
from streaming import StreamingDataset
from streaming.base.converters import dataframe_to_mds
from streaming.base import MDSWriter
import os

# Parameters required for saving data in MDS format
dense_dict = { key: 'float64' for key in dense_cols }
cat_dict = { key: 'int64' for key in cat_cols }
label_dict = { 'label' : 'int64' }
columns = {**label_dict, **dense_dict, **cat_dict}

compression = 'zstd:7'

# 修改为您的 GCS 存储路径
bucket_path = "gs://project-kangwe-poc-dlrm/mds_data"
output_dir_train = f"{bucket_path}/train"
output_dir_validation = f"{bucket_path}/validation"
output_dir_test = f"{bucket_path}/test"

def save_data(df, output_path, label, num_workers=20):
    print(f"Saving {label} data to GCS: {output_path}")
    # dataframe_to_mds 直接支持 gs:// 路径
    mds_kwargs = {'out': output_path, 'columns': columns, 'compression': compression}
    dataframe_to_mds(df.repartition(num_workers), merge_index=True, mds_kwargs=mds_kwargs)

# 执行保存
save_data(train_df, output_dir_train, 'train')
save_data(validation_df, output_dir_validation, 'validation')
save_data(test_df, output_dir_test, 'test')

# Step 2. Helper Functions for Recommendation Dataloading
In this step, you install the necessary libraries, add imports, and add some relevant helper functions to train the model.

# 2.1. Installs and Imports

In [ ]:
# Fix dependency conflicts: Force install compatible versions to ensure binary ABI consistency
%pip install -q --upgrade "numpy<2.0.0"
%pip install -q --upgrade --no-cache-dir \
    torch==2.2.2 \
    torchvision==0.17.2 \
    torchaudio==2.2.2 \
    torchrec==0.6.0 \
    fbgemm-gpu==0.6.0 \
    --index-url https://download.pytorch.org/whl/cu118

# Install TorchRec, FBGEMM, and explicitly include transformers to resolve torchmetrics dependency issues
%pip install -q --no-cache-dir \
    torchmetrics==1.0.3 \
    iopath==0.1.10 \
    pyre_extensions==0.0.32 \
    mosaicml-streaming==0.7.5 \
    "transformers==4.44.2"

In [ ]:
# Before execute this cell, restart the session to load the CUDA 11
import contextlib
import tempfile
from typing import Generator
import csv
import os
import random
import sys

# Fix: Explicitly check for CUDA and try to load torch before others
try:
    import torch
    print(f"Torch version: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
except ImportError as e:
    print(f"Failed to import torch: {e}")

# The OSError for torchaudio usually stems from missing libcudart.so.11.0
# If you are on a CUDA 12 system, you might need to symlink or install the specific toolkit
import torchaudio

from streaming import StreamingDataset, StreamingDataLoader
from streaming.base import MDSWriter

import torch.distributed as dist
from torch.utils.data import DataLoader
import itertools
import torchmetrics as metrics
from tqdm import tqdm
from dataclasses import dataclass, field
from typing import List, Optional

from torchrec import EmbeddingBagCollection
from torchrec.distributed import TrainPipelineSparseDist
from torchrec.distributed.comm import get_local_size
from torchrec.distributed.model_parallel import (
    DistributedModelParallel,
    get_default_sharders,
)
from torchrec.distributed.planner import EmbeddingShardingPlanner, Topology
from torchrec.distributed.planner.storage_reservations import (
    HeuristicalStorageReservation,
)
from torchrec.models.dlrm import DLRM, DLRM_DCN, DLRM_Projection, DLRMTrain
from torchrec.optim.apply_optimizer_in_backward import apply_optimizer_in_backward
from torchrec.modules.embedding_configs import EmbeddingBagConfig
from torchrec.optim.keyed import CombinedOptimizer, KeyedOptimizerWrapper
from torchrec.optim.optimizers import in_backward_optimizer_filter

from torch.optim.lr_scheduler import _LRScheduler
from pyre_extensions import none_throws
from torchrec.datasets.utils import Batch
from torchrec.sparse.jagged_tensor import KeyedJaggedTensor
from typing import Optional, List, Iterable, cast
from collections import defaultdict
from functools import partial
import dataclasses
from dataclasses import asdict

from torch import nn
from torch.distributed._sharded_tensor import ShardedTensor

from pyspark.ml.torch.distributor import TorchDistributor

# 2.2. Helper functions for Converting to Pipelineable DataType
Using TorchRec pipelines requires a pipelineable data type (which is Batch in this case). In this step, you create a helper function that takes each batch from the StreamingDataset and passes it through a transformation function to convert it into a pipelineable batch.

For further context, see:

https://github.com/pytorch/torchrec/blob/29f503a8855040bc49788d3ad24e7ab93d944885/torchrec/datasets/utils.py#L28

In [ ]:
# TODO: This is from earlier outputs (section 1.2, cell 2); if another dataset is being used, these values need to be updated
# {'cat_1': 103, 'cat_2': 180, 'cat_3': 93, 'cat_4': 15, 'cat_5': 107, 'cat_6': 72, 'cat_7': 189, 'cat_8': 21, 'cat_9': 103, 'cat_10': 122}
cat_data = {'cat_1': 103, 'cat_2': 180, 'cat_3': 93, 'cat_4': 15, 'cat_5': 107, 'cat_6': 72, 'cat_7': 189, 'cat_8': 21, 'cat_9': 103, 'cat_10': 122}
dense_cols = [f"int_{i}" for i in range(1, 11)]
cat_cols = [f"cat_{i}" for i in range(1, 11)]
emb_counts = [cat_data[k] for k in cat_cols]

def transform_to_torchrec_batch(batch, num_embeddings_per_feature: Optional[List[int]] = None) -> Batch:
    cat_list = []
    for col_name in dense_cols:
        val = torch.tensor(batch[col_name], dtype=torch.float32)
        cat_list.append(val.unsqueeze(0).T)
    dense_features = torch.cat(cat_list, dim=1)

    kjt_values: List[int] = []
    kjt_lengths: List[int] = []
    for col_idx, col_name in enumerate(cat_cols):
        values = batch[col_name]
        for value in values:
            if value:
                kjt_values.append(
                    value % num_embeddings_per_feature[col_idx]
                )
                kjt_lengths.append(1)
            else:
                kjt_lengths.append(0)

    sparse_features = KeyedJaggedTensor.from_lengths_sync(
        cat_cols,
        torch.tensor(kjt_values),
        torch.tensor(kjt_lengths, dtype=torch.int32),
    )
    labels = torch.tensor(batch["label"], dtype=torch.int32)
    assert isinstance(labels, torch.Tensor)

    return Batch(
        dense_features=dense_features,
        sparse_features=sparse_features,
        labels=labels,
    )

transform_partial = partial(transform_to_torchrec_batch, num_embeddings_per_feature=emb_counts)

# 2.3. Helper Function for DataLoading using Mosaic's StreamingDataset
This utilizes Mosaic's StreamingDataset and Mosaic's StreamingDataLoader for efficient data loading. For more information, view this documentation.

In [ ]:
def get_dataloader_with_mosaic(remote_path, local_cache_path, batch_size, label):
    """
    支持从 GCS (remote_path) 流式读取数据，并缓存到本地 (local_cache_path)
    """
    print(f"Initializing {label} streaming dataset from: {remote_path}")
    dataset = StreamingDataset(
        remote=remote_path,
        local=local_cache_path,
        shuffle=True,
        batch_size=batch_size
    )
    return StreamingDataLoader(dataset, batch_size=batch_size)

# Step 3. Creating the Relevant TorchRec code for Training
This contains all of the training and evaluation code.

# 3.1. Base Dataclass for Training inputs
Feel free to modify any of the variables mentioned here, but note that the final layer for dense_arch_layer_sizes should be equivalent to embedding_dim.

In [ ]:
# TODO: Update these values for training as needed
@dataclass
class Args:
    """
    Training arguments.
    """
    epochs: int = 3  # Training for one Epoch
    embedding_dim: int = 128  # Embedding dimension is 128
    dense_arch_layer_sizes: list = dataclasses.field(default_factory=lambda: [512, 256, 128])  # The last layer for dense should be equivalent to `embedding_dim`
    over_arch_layer_sizes: list = dataclasses.field(default_factory=lambda: [512, 512, 256, 1])  # We are essentially doing binary classification here, so the last layer is 1
    learning_rate: float = 0.03
    eps: float = 1e-8
    batch_size: int = 512
    print_sharding_plan: bool = True
    print_lr: bool = False  # Optional, prints the learning rate at each iteration step
    lr_warmup_steps: int = 0  # Optional, sets the learning rate warmup steps
    lr_decay_start: int = 0  # Optional, sets the decay start for learning rate
    lr_decay_steps: int = 0  # Optional, sets the decay steps for learning rate
    validation_freq: int = None  # Optional, determines how often during training you want to run validation (# of training steps)
    limit_train_batches: int = None  # Optional, limits the number of training batches
    limit_val_batches: int = None  # Optional, limits the number of validation batches
    limit_test_batches: int = None  # Optional, limits the number of test batches

# We will use this to store our results in mlflow
def get_relevant_fields(args, dense_cols, cat_cols, emb_counts):
    fields_to_save = ["epochs", "embedding_dim", "dense_arch_layer_sizes", "over_arch_layer_sizes", "learning_rate", "eps", "batch_size"]
    result = { key: getattr(args, key) for key in fields_to_save }
    # adding dense cols
    result["dense_cols"] = dense_cols
    result["cat_cols"] = cat_cols
    result["emb_counts"] = emb_counts
    return result

def to_vertex_params(d):
    """Vertex AI log_params 只接受 int/float/bool/str；其余类型转成 str 以保留可读性。"""
    return {k: (v if isinstance(v, (int, float, bool, str)) else str(v)) for k, v in d.items()}

In [ ]:
@dataclass
class TrainValTestResults:
    """
    A dataclass to store our results.
    """
    val_aurocs: List[float] = field(default_factory=list)
    test_auroc: Optional[float] = None

# 3.2. LR Scheduler
This isn't specifically used unless you want to schedule the learning rate for the Adagrad Optimizer.

In [ ]:
class LRPolicyScheduler(_LRScheduler):
    def __init__(self, optimizer, num_warmup_steps, decay_start_step, num_decay_steps):
        self.num_warmup_steps = num_warmup_steps
        self.decay_start_step = decay_start_step
        self.decay_end_step = decay_start_step + num_decay_steps
        self.num_decay_steps = num_decay_steps

        if self.decay_start_step < self.num_warmup_steps:
            sys.exit("Learning rate warmup must finish before the decay starts")

        super(LRPolicyScheduler, self).__init__(optimizer)

    def get_lr(self):
        step_count = self._step_count
        if step_count < self.num_warmup_steps:
            # warmup
            scale = 1.0 - (self.num_warmup_steps - step_count) / self.num_warmup_steps
            lr = [base_lr * scale for base_lr in self.base_lrs]
            self.last_lr = lr
        elif self.decay_start_step <= step_count and step_count < self.decay_end_step:
            # decay
            decayed_steps = step_count - self.decay_start_step
            scale = ((self.num_decay_steps - decayed_steps) / self.num_decay_steps) ** 2
            min_lr = 0.0000001
            lr = [max(min_lr, base_lr * scale) for base_lr in self.base_lrs]
            self.last_lr = lr
        else:
            if self.num_decay_steps > 0:
                # freeze at last, either because we're after decay
                # or because we're between warmup and decay
                lr = self.last_lr
            else:
                # do not adjust
                lr = self.base_lrs
        return lr

# 3.3. Training and Evaluation Helper Functions

In [ ]:
def batched(it, n):
    assert n >= 1
    for x in it:
        yield itertools.chain((x,), itertools.islice(it, n - 1))

# 3.4.1. Helper Functions for Distributed Model Saving

In [ ]:
# DLRM and TorchRec use special tensors called ShardedTensors, so we localize them
# and put them in the same rank that is saving to mlflow
def gather_and_get_state_dict(model):
    rank = dist.get_rank()
    world_size = dist.get_world_size()
    state_dict = model.state_dict()
    gathered_state_dict = {}

    # Iterate over all items in the state_dict
    for fqn, tensor in state_dict.items():
        if isinstance(tensor, ShardedTensor):
            # Collect all shards of the tensor across ranks
            full_tensor = None
            if rank == 0:
                full_tensor = torch.zeros(tensor.size()).to(tensor.device)
            tensor.gather(0, full_tensor)
            if rank == 0:
                gathered_state_dict[fqn] = full_tensor
        else:
            # Directly add non-sharded tensors to the new state_dict
            if rank == 0:
                gathered_state_dict[fqn] = tensor

    return gathered_state_dict

In [ ]:
from google.cloud import aiplatform
import os

# 初始化 Vertex AI Experiment
PROJECT_ID = "project-kangwe-poc"
REGION = "us-central1"
EXPERIMENT_NAME = "dlrm-training-demo"

aiplatform.init(project=PROJECT_ID, location=REGION, experiment=EXPERIMENT_NAME)

def log_metrics_gcp(metrics_dict, step=None):
    try:
        from google.cloud import aiplatform
        if step is not None:
            aiplatform.log_time_series_metrics(metrics_dict, step=step)
        else:
            aiplatform.log_metrics(metrics_dict)
    except Exception as e:
        print(f"[warn] vertex log failed: {e}")

In [ ]:
def log_state_dict_to_gcs(model, epoch, bucket_path, run_name=None):
    state_dict = gather_and_get_state_dict(model)
    if dist.get_rank() != 0 or not state_dict:
        return None

    from google.cloud import storage, aiplatform

    local_path = f"/tmp/checkpoint_epoch_{epoch}.pt"
    torch.save(state_dict, local_path)
    bucket_name = bucket_path.split("/")[2]
    prefix = "/".join(bucket_path.split("/")[3:]).strip("/")
    subdir = f"models/{run_name}" if run_name else "models"
    blob_path = "/".join(p for p in [prefix, subdir, f"checkpoint_epoch_{epoch}.pt"] if p)
    storage.Client().bucket(bucket_name).blob(blob_path).upload_from_filename(local_path)
    os.remove(local_path)
    gcs_uri = f"gs://{bucket_name}/{blob_path}"
    print(f"Model saved to {gcs_uri}")

    # 把 checkpoint 作为 system.Model Artifact 记入 Vertex AI Metadata，并把 URI 写进当前 run 的 params
    try:
        aiplatform.Artifact.create(
            schema_title="system.Model",
            display_name=f"dlrm-ckpt-epoch-{epoch}" + (f"-{run_name}" if run_name else ""),
            uri=gcs_uri,
            metadata={"epoch": epoch, "framework": "pytorch-torchrec"},
        )
        aiplatform.log_params({f"checkpoint_epoch_{epoch}_uri": gcs_uri})
    except Exception as e:
        print(f"[warn] register model artifact failed: {e}")
    return gcs_uri

# 3.4.2. Helper Functions for Distributed Model Training and Evaluation

In [ ]:
def train(
    pipeline: TrainPipelineSparseDist,
    train_dataloader: DataLoader,
    val_dataloader: DataLoader,
    epoch: int,
    lr_scheduler,
    print_lr: bool,
    validation_freq: Optional[int],
    limit_train_batches: Optional[int],
    limit_val_batches: Optional[int]) -> None:
    """
    Trains model for 1 epoch. Helper function for train_val_test.

    Args:
        pipeline (TrainPipelineSparseDist): data pipeline.
        train_dataloader (DataLoader): Training set's dataloader.
        val_dataloader (DataLoader): Validation set's dataloader.
        epoch (int): The number of complete passes through the training set so far.
        lr_scheduler (LRPolicyScheduler): Learning rate scheduler.
        print_lr (bool): Whether to print the learning rate every training step.
        validation_freq (Optional[int]): The number of training steps between validation runs within an epoch.
        limit_train_batches (Optional[int]): Limits the training set to the first `limit_train_batches` batches.
        limit_val_batches (Optional[int]): Limits the validation set to the first `limit_val_batches` batches.

    Returns:
        None.
    """
    pipeline._model.train()

    # Get the first `limit_train_batches` batches
    iterator = itertools.islice(iter(train_dataloader), limit_train_batches)

    # Only print out the progress bar on rank 0
    is_rank_zero = dist.get_rank() == 0
    if is_rank_zero:
        pbar = tqdm(
            iter(int, 1),
            desc=f"Epoch {epoch}",
            total=len(train_dataloader),
            disable=False,
        )

    # TorchRec's pipeline paradigm is unique as it takes in an iterator of batches for training.
    start_it = 0
    n = validation_freq if validation_freq else len(train_dataloader)
    for batched_iterator in batched(iterator, n):
        for it in itertools.count(start_it):
            try:
                if is_rank_zero and print_lr:
                    for i, g in enumerate(pipeline._optimizer.param_groups):
                        print(f"lr: {it} {i} {g['lr']:.6f}")
                pipeline.progress(map(transform_partial, batched_iterator))
                lr_scheduler.step()
                if is_rank_zero:
                    pbar.update(1)
            except StopIteration:
                if is_rank_zero:
                    print("Total number of iterations:", it)
                start_it = it
                break

        # If we are validating frequently, use the evaluation function
        if validation_freq and start_it % validation_freq == 0:
            evaluate(limit_val_batches, pipeline, val_dataloader, "val")
            pipeline._model.train()

In [ ]:
def evaluate(
    limit_batches: Optional[int],
    pipeline: TrainPipelineSparseDist,
    eval_dataloader: DataLoader,
    stage: str) -> float:
    """
    Evaluates model. Computes and prints AUROC. Helper function for train_val_test.

    Args:
        limit_batches (Optional[int]): Limits the dataloader to the first `limit_batches` batches.
        pipeline (TrainPipelineSparseDist): data pipeline.
        eval_dataloader (DataLoader): Dataloader for either the validation set or test set.
        stage (str): "val" or "test".

    Returns:
        float: auroc result
    """
    pipeline._model.eval()
    device = pipeline._device

    iterator = itertools.islice(iter(eval_dataloader), limit_batches)

    # We are using AUROC for our metric
    auroc = metrics.AUROC(task="binary").to(device)

    is_rank_zero = dist.get_rank() == 0
    if is_rank_zero:
        pbar = tqdm(
            iter(int, 1),
            desc=f"Evaluating {stage} set",
            total=len(eval_dataloader),
            disable=False,
        )
    with torch.no_grad():
        while True:
            try:
                _loss, logits, labels = pipeline.progress(map(transform_partial, iterator))
                preds = torch.sigmoid(logits)
                auroc(preds, labels)
                if is_rank_zero:
                    pbar.update(1)
            except StopIteration:
                break

    auroc_result = auroc.compute().item()
    num_samples = torch.tensor(sum(map(len, auroc.target)), device=device)
    dist.reduce(num_samples, 0, op=dist.ReduceOp.SUM)

    if is_rank_zero:
        print(f"AUROC over {stage} set: {auroc_result}.")
        print(f"Number of {stage} samples: {num_samples}")
    return auroc_result

In [ ]:
def train_val_test(args, model, optimizer, device, train_dataloader, val_dataloader, test_dataloader, lr_scheduler, bucket_path=None, run_name=None) -> TrainValTestResults:
    """
    Train/validation/test loop updated for GCP Vertex AI and GCS.
    """
    results = TrainValTestResults()
    pipeline = TrainPipelineSparseDist(model, optimizer, device, execute_all_batches=True)

    # Getting base auroc and logging to Vertex AI / MLflow
    val_auroc = evaluate(args.limit_val_batches, pipeline, val_dataloader, "val")
    results.val_aurocs.append(val_auroc)
    if int(os.environ["RANK"]) == 0:
        log_metrics_gcp({'val_auroc': val_auroc}, step=0)

    last_ckpt_uri = None

    # Running a training loop
    for epoch in range(args.epochs):
        train(
            pipeline,
            train_dataloader,
            val_dataloader,
            epoch,
            lr_scheduler,
            args.print_lr,
            args.validation_freq,
            args.limit_train_batches,
            args.limit_val_batches,
        )

        # Evaluate after each training epoch
        val_auroc = evaluate(args.limit_val_batches, pipeline, val_dataloader, "val")
        results.val_aurocs.append(val_auroc)
        if int(os.environ["RANK"]) == 0:
            log_metrics_gcp({'val_auroc': val_auroc}, step=epoch + 1)

        # Save the model state dict to GCS instead of MLflow
        if bucket_path:
            last_ckpt_uri = log_state_dict_to_gcs(pipeline._model.module, epoch, bucket_path, run_name=run_name)

    # Evaluate on the test set after training loop finishes
    test_auroc = evaluate(args.limit_test_batches, pipeline, test_dataloader, "test")
    results.test_auroc = test_auroc
    if int(os.environ["RANK"]) == 0:
        log_metrics_gcp({'test_auroc': test_auroc})
        if last_ckpt_uri:
            try:
                from google.cloud import aiplatform
                aiplatform.log_params({"final_model_uri": last_ckpt_uri})
            except Exception as e:
                print(f"[warn] log final_model_uri failed: {e}")

    return results

# 3.6. The Main Function
This function will train the DLRM recommendation model. For more information, view the following guides/docs/code:

https://pytorch.org/torchrec/
https://github.com/pytorch/torchrec
https://github.com/facebookresearch/dlrm/blob/main/torchrec_dlrm/dlrm_main.py

In [ ]:
# TODO: Specify where the data is stored in UC Volumes
input_dir_train = "gs://project-kangwe-poc-dlrm/mds_data/train"
input_dir_validation = "gs://project-kangwe-poc-dlrm/mds_data/validation"
input_dir_test = "gs://project-kangwe-poc-dlrm/mds_data/test"

def main(args):
    import torch
    from google.cloud import aiplatform

    # Some preliminary torch setup
    torch.jit._state.disable()
    global_rank = int(os.environ["RANK"])
    local_rank = int(os.environ["LOCAL_RANK"])
    device = torch.device(f"cuda:{local_rank}")
    backend = "nccl"
    torch.cuda.set_device(device)

    # 在主进程中开启 Vertex AI Experiment Run
    if global_rank == 0:
        run_name = f"run-{pd.Timestamp.now().strftime('%Y%m%d-%H%M%S')}"
        aiplatform.start_run(run=run_name)
        param_dict = get_relevant_fields(args, dense_cols, cat_cols, emb_counts)
        aiplatform.log_params(to_vertex_params(param_dict))

    # Starting distributed process group
    dist.init_process_group(backend=backend)

    # Loading the data
    train_dataloader = get_dataloader_with_mosaic(input_dir_train, "/tmp/cache/train", args.batch_size, "train")
    val_dataloader = get_dataloader_with_mosaic(input_dir_validation, "/tmp/cache/val", args.batch_size, "val")
    test_dataloader = get_dataloader_with_mosaic(input_dir_test, "/tmp/cache/test", args.batch_size, "test")

    # Creating the embedding tables
    eb_configs = [
        EmbeddingBagConfig(
            name=f"t_{feature_name}",
            embedding_dim=args.embedding_dim,
            num_embeddings=emb_counts[feature_idx],
            feature_names=[feature_name],
        )
        for feature_idx, feature_name in enumerate(cat_cols)
    ]

    dlrm_model = DLRM(
            embedding_bag_collection=EmbeddingBagCollection(
                tables=eb_configs, device=torch.device("meta")
            ),
            dense_in_features=len(dense_cols),
            dense_arch_layer_sizes=args.dense_arch_layer_sizes,
            over_arch_layer_sizes=args.over_arch_layer_sizes,
            dense_device=device,
        )

    train_model = DLRMTrain(dlrm_model)

    # Sharding and Parallelism
    planner = EmbeddingShardingPlanner(
        topology=Topology(
            local_world_size=get_local_size(),
            world_size=dist.get_world_size(),
            compute_device=device.type,
        ),
        batch_size=args.batch_size,
        storage_reservation=HeuristicalStorageReservation(percentage=0.05),
    )
    plan = planner.collective_plan(train_model, get_default_sharders(), dist.GroupMember.WORLD)
    model = DistributedModelParallel(module=train_model, device=device, plan=plan)

    # Optimizer setup
    dense_optimizer = KeyedOptimizerWrapper(
        dict(in_backward_optimizer_filter(model.named_parameters())),
        lambda params: torch.optim.Adagrad(params, lr=args.learning_rate, eps=args.eps),
    )
    optimizer = CombinedOptimizer([model.fused_optimizer, dense_optimizer])
    lr_scheduler = LRPolicyScheduler(optimizer, args.lr_warmup_steps, args.lr_decay_start, args.lr_decay_steps)

    results = train_val_test(
        args, model, optimizer, device,
        train_dataloader, val_dataloader, test_dataloader,
        lr_scheduler, bucket_path="gs://project-kangwe-poc-dlrm",
        run_name=run_name
    )

    if global_rank == 0:
        aiplatform.end_run()

    dist.destroy_process_group()

# Step 4. Single Node + Single GPU Training
Here, you set the environment variables to run training over the sample set of 100,000 data points (stored in Volumes in Unity Catalog and collected using Mosaic StreamingDataset). You can expect each epoch to take ~40 minutes.



In [ ]:
import torch.distributed as dist
import os
import shutil
import pandas as pd
from streaming.base.util import clean_stale_shared_memory

# 设置环境变量
os.environ["RANK"] = "0"
os.environ["LOCAL_RANK"] = "0"
os.environ["WORLD_SIZE"] = "1"
os.environ["MASTER_ADDR"] = "localhost"
os.environ["MASTER_PORT"] = "29500"

# 修复 1：清理 StreamingDataset 残留的共享内存
clean_stale_shared_memory()

# 修复 2：清理之前的本地缓存目录，避免 "Reused local directory" 错误
cache_dir = "/tmp/cache"
if os.path.exists(cache_dir):
    shutil.rmtree(cache_dir)

# 修复 3：如果进程组已存在，先销毁它，避免 "initialize twice" 错误
if dist.is_initialized():
    dist.destroy_process_group()

args = Args()
main(args)

# Step 5. Single Node + Multi GPU Training
This notebook uses the TorchDistributor for handling training on a g4dn.12xlarge instance with 4 T4 GPUs. You can view the sharding plan to see what tables are located on what GPUs. This takes ~14 minutes to run per epoch.

Note: There may be cases where you receive unexpected errors (like the Python Kernel crashing or segmentation faults). This is a transient error and the easiest way to overcome it is to skip the single node single GPU training code before we run any distributed code (single node multi GPU or multi node multi GPU).

Note: If you see any logs that are associated with Mosaic Data Loading, these are transient errors that can be overcome by simply rerunning the failed cell.

In [ ]:
BUILD_DIR = "vertex_build"

import os
os.makedirs(BUILD_DIR, exist_ok=True)

In [ ]:
%%writefile {BUILD_DIR}/train_dlrm_multigpu.py
#!/usr/bin/env python
"""
DLRM multi-GPU training (TorchRec + Mosaic StreamingDataset).

Launch with:
    torchrun --standalone --nproc_per_node=4 train_dlrm_multigpu.py
"""
import os
import sys
import shutil
import itertools
import dataclasses
from dataclasses import dataclass, field
from functools import partial
from typing import List, Optional
from datetime import datetime

import torch
import torch.distributed as dist
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import _LRScheduler
from torch.distributed._sharded_tensor import ShardedTensor

import torchmetrics as metrics
from tqdm import tqdm

from streaming import StreamingDataset, StreamingDataLoader
from streaming.base.util import clean_stale_shared_memory

from torchrec import EmbeddingBagCollection
from torchrec.distributed import TrainPipelineSparseDist
from torchrec.distributed.comm import get_local_size
from torchrec.distributed.model_parallel import DistributedModelParallel, get_default_sharders
from torchrec.distributed.planner import EmbeddingShardingPlanner, Topology
from torchrec.distributed.planner.storage_reservations import HeuristicalStorageReservation
from torchrec.models.dlrm import DLRM, DLRMTrain
from torchrec.modules.embedding_configs import EmbeddingBagConfig
from torchrec.optim.keyed import CombinedOptimizer, KeyedOptimizerWrapper
from torchrec.optim.optimizers import in_backward_optimizer_filter
from torchrec.datasets.utils import Batch
from torchrec.sparse.jagged_tensor import KeyedJaggedTensor


# ----------------------------------------------------------------------------
# Config: dataset schema & paths (from notebook section 1.2 output)
# ----------------------------------------------------------------------------
CAT_DATA = {'cat_1': 103, 'cat_2': 180, 'cat_3': 93, 'cat_4': 15, 'cat_5': 107,
            'cat_6': 72, 'cat_7': 189, 'cat_8': 21, 'cat_9': 103, 'cat_10': 122}
DENSE_COLS = [f"int_{i}" for i in range(1, 11)]
CAT_COLS = [f"cat_{i}" for i in range(1, 11)]
EMB_COUNTS = [CAT_DATA[k] for k in CAT_COLS]

INPUT_DIR_TRAIN = "gs://project-kangwe-poc-dlrm/mds_data/train"
INPUT_DIR_VALIDATION = "gs://project-kangwe-poc-dlrm/mds_data/validation"
INPUT_DIR_TEST = "gs://project-kangwe-poc-dlrm/mds_data/test"
BUCKET_PATH = "gs://project-kangwe-poc-dlrm"
LOCAL_CACHE_ROOT = "/tmp/cache"

PROJECT_ID = "project-kangwe-poc"
REGION = "us-central1"
EXPERIMENT_NAME = "dlrm-training-demo"


# ----------------------------------------------------------------------------
# Args
# ----------------------------------------------------------------------------
@dataclass
class Args:
    epochs: int = 3
    embedding_dim: int = 128
    dense_arch_layer_sizes: list = field(default_factory=lambda: [512, 256, 128])
    over_arch_layer_sizes: list = field(default_factory=lambda: [512, 512, 256, 1])
    learning_rate: float = 0.03
    eps: float = 1e-8
    batch_size: int = 512
    print_sharding_plan: bool = True
    print_lr: bool = False
    lr_warmup_steps: int = 0
    lr_decay_start: int = 0
    lr_decay_steps: int = 0
    validation_freq: Optional[int] = None
    limit_train_batches: Optional[int] = None
    limit_val_batches: Optional[int] = None
    limit_test_batches: Optional[int] = None


@dataclass
class TrainValTestResults:
    val_aurocs: List[float] = field(default_factory=list)
    test_auroc: Optional[float] = None


def get_relevant_fields(args):
    keys = ["epochs", "embedding_dim", "dense_arch_layer_sizes",
            "over_arch_layer_sizes", "learning_rate", "eps", "batch_size"]
    result = {k: getattr(args, k) for k in keys}
    result["dense_cols"] = DENSE_COLS
    result["cat_cols"] = CAT_COLS
    result["emb_counts"] = EMB_COUNTS
    return result


def to_vertex_params(d):
    """Vertex AI log_params 只接受 int/float/bool/str；其余类型转成 str 以保留可读性。"""
    return {k: (v if isinstance(v, (int, float, bool, str)) else str(v)) for k, v in d.items()}


# ----------------------------------------------------------------------------
# Data loading
# ----------------------------------------------------------------------------
def transform_to_torchrec_batch(batch, num_embeddings_per_feature: List[int]) -> Batch:
    cat_list = []
    for col_name in DENSE_COLS:
        val = batch[col_name].clone().detach().to(torch.float32)
        cat_list.append(val.unsqueeze(0).T)
    dense_features = torch.cat(cat_list, dim=1)

    kjt_values: List[int] = []
    kjt_lengths: List[int] = []
    for col_idx, col_name in enumerate(CAT_COLS):
        values = batch[col_name]
        for value in values:
            if value:
                kjt_values.append(int(value) % num_embeddings_per_feature[col_idx])
                kjt_lengths.append(1)
            else:
                kjt_lengths.append(0)

    sparse_features = KeyedJaggedTensor.from_lengths_sync(
        CAT_COLS,
        torch.tensor(kjt_values),
        torch.tensor(kjt_lengths, dtype=torch.int32),
    )
    labels = batch["label"].clone().detach().to(torch.int32)
    return Batch(dense_features=dense_features, sparse_features=sparse_features, labels=labels)


transform_partial = partial(transform_to_torchrec_batch, num_embeddings_per_feature=EMB_COUNTS)


def get_dataloader_with_mosaic(remote_path, local_cache_path, batch_size, label):
    print(f"[rank {os.environ.get('RANK', '?')}] Initializing {label} streaming dataset from: {remote_path}")
    dataset = StreamingDataset(
        remote=remote_path,
        local=local_cache_path,
        shuffle=True,
        batch_size=batch_size,
        download_timeout=300,
    )
    return StreamingDataLoader(dataset, batch_size=batch_size)


# ----------------------------------------------------------------------------
# LR scheduler
# ----------------------------------------------------------------------------
class LRPolicyScheduler(_LRScheduler):
    def __init__(self, optimizer, num_warmup_steps, decay_start_step, num_decay_steps):
        self.num_warmup_steps = num_warmup_steps
        self.decay_start_step = decay_start_step
        self.decay_end_step = decay_start_step + num_decay_steps
        self.num_decay_steps = num_decay_steps
        if self.decay_start_step < self.num_warmup_steps:
            sys.exit("Learning rate warmup must finish before the decay starts")
        super().__init__(optimizer)

    def get_lr(self):
        step_count = self._step_count
        if step_count < self.num_warmup_steps:
            scale = 1.0 - (self.num_warmup_steps - step_count) / self.num_warmup_steps
            lr = [base_lr * scale for base_lr in self.base_lrs]
            self.last_lr = lr
        elif self.decay_start_step <= step_count < self.decay_end_step:
            decayed_steps = step_count - self.decay_start_step
            scale = ((self.num_decay_steps - decayed_steps) / self.num_decay_steps) ** 2
            min_lr = 1e-7
            lr = [max(min_lr, base_lr * scale) for base_lr in self.base_lrs]
            self.last_lr = lr
        else:
            lr = self.last_lr if self.num_decay_steps > 0 else self.base_lrs
        return lr


# ----------------------------------------------------------------------------
# Checkpoint / metrics logging
# ----------------------------------------------------------------------------
def gather_and_get_state_dict(model):
    rank = dist.get_rank()
    state_dict = model.state_dict()
    gathered = {}
    for fqn, tensor in state_dict.items():
        if isinstance(tensor, ShardedTensor):
            full_tensor = torch.zeros(tensor.size()).to(tensor.device) if rank == 0 else None
            tensor.gather(0, full_tensor)
            if rank == 0:
                gathered[fqn] = full_tensor
        elif rank == 0:
            gathered[fqn] = tensor
    return gathered


def log_state_dict_to_gcs(model, epoch, bucket_path, run_name=None):
    state_dict = gather_and_get_state_dict(model)
    if dist.get_rank() != 0 or not state_dict:
        return None

    from google.cloud import storage, aiplatform

    local_path = f"/tmp/checkpoint_epoch_{epoch}.pt"
    torch.save(state_dict, local_path)
    bucket_name = bucket_path.split("/")[2]
    prefix = "/".join(bucket_path.split("/")[3:]).strip("/")
    subdir = f"models/{run_name}" if run_name else "models"
    blob_path = "/".join(p for p in [prefix, subdir, f"checkpoint_epoch_{epoch}.pt"] if p)
    storage.Client().bucket(bucket_name).blob(blob_path).upload_from_filename(local_path)
    os.remove(local_path)
    gcs_uri = f"gs://{bucket_name}/{blob_path}"
    print(f"Model saved to {gcs_uri}")

    # 把 checkpoint 作为 system.Model Artifact 记入 Vertex AI Metadata，并把 URI 写进当前 run 的 params
    try:
        aiplatform.Artifact.create(
            schema_title="system.Model",
            display_name=f"dlrm-ckpt-epoch-{epoch}" + (f"-{run_name}" if run_name else ""),
            uri=gcs_uri,
            metadata={"epoch": epoch, "framework": "pytorch-torchrec"},
        )
        aiplatform.log_params({f"checkpoint_epoch_{epoch}_uri": gcs_uri})
    except Exception as e:
        print(f"[warn] register model artifact failed: {e}")
    return gcs_uri


def log_metrics_gcp(metrics_dict, step=None):
    try:
        from google.cloud import aiplatform
        if step is not None:
            aiplatform.log_time_series_metrics(metrics_dict, step=step)
        else:
            aiplatform.log_metrics(metrics_dict)
    except Exception as e:
        print(f"[warn] vertex log failed: {e}")


# ----------------------------------------------------------------------------
# Train / eval loops
# ----------------------------------------------------------------------------
def batched(it, n):
    assert n >= 1
    for x in it:
        yield itertools.chain((x,), itertools.islice(it, n - 1))


def train_one_epoch(pipeline, train_dl, val_dl, epoch, lr_scheduler, args):
    pipeline._model.train()
    iterator = itertools.islice(iter(train_dl), args.limit_train_batches)
    is_rank_zero = dist.get_rank() == 0
    pbar = tqdm(iter(int, 1), desc=f"Epoch {epoch}", total=len(train_dl), disable=not is_rank_zero)

    start_it = 0
    n = args.validation_freq if args.validation_freq else len(train_dl)
    for batched_iterator in batched(iterator, n):
        for it in itertools.count(start_it):
            try:
                if is_rank_zero and args.print_lr:
                    for i, g in enumerate(pipeline._optimizer.param_groups):
                        print(f"lr: {it} {i} {g['lr']:.6f}")
                pipeline.progress(map(transform_partial, batched_iterator))
                lr_scheduler.step()
                pbar.update(1)
            except StopIteration:
                if is_rank_zero:
                    print(f"Total number of iterations: {it}")
                start_it = it
                break
        if args.validation_freq and start_it % args.validation_freq == 0:
            evaluate(args.limit_val_batches, pipeline, val_dl, "val")
            pipeline._model.train()
    pbar.close()


def evaluate(limit_batches, pipeline, eval_dl, stage):
    pipeline._model.eval()
    device = pipeline._device
    iterator = itertools.islice(iter(eval_dl), limit_batches)
    auroc = metrics.AUROC(task="binary").to(device)
    is_rank_zero = dist.get_rank() == 0
    pbar = tqdm(iter(int, 1), desc=f"Evaluating {stage} set", total=len(eval_dl), disable=not is_rank_zero)

    with torch.no_grad():
        while True:
            try:
                _loss, logits, labels = pipeline.progress(map(transform_partial, iterator))
                preds = torch.sigmoid(logits)
                auroc(preds, labels)
                pbar.update(1)
            except StopIteration:
                break
    pbar.close()

    auroc_result = auroc.compute().item()
    num_samples = torch.tensor(sum(map(len, auroc.target)), device=device)
    dist.reduce(num_samples, 0, op=dist.ReduceOp.SUM)
    if is_rank_zero:
        print(f"AUROC over {stage} set: {auroc_result}.")
        print(f"Number of {stage} samples: {num_samples}")
    return auroc_result


def train_val_test(args, model, optimizer, device, train_dl, val_dl, test_dl, lr_scheduler, run_name=None):
    results = TrainValTestResults()
    pipeline = TrainPipelineSparseDist(model, optimizer, device, execute_all_batches=True)
    is_rank_zero = dist.get_rank() == 0

    val_auroc = evaluate(args.limit_val_batches, pipeline, val_dl, "val")
    results.val_aurocs.append(val_auroc)
    if is_rank_zero:
        log_metrics_gcp({'val_auroc': val_auroc}, step=0)

    last_ckpt_uri = None
    for epoch in range(args.epochs):
        train_one_epoch(pipeline, train_dl, val_dl, epoch, lr_scheduler, args)
        val_auroc = evaluate(args.limit_val_batches, pipeline, val_dl, "val")
        results.val_aurocs.append(val_auroc)
        if is_rank_zero:
            log_metrics_gcp({'val_auroc': val_auroc}, step=epoch + 1)
        last_ckpt_uri = log_state_dict_to_gcs(pipeline._model.module, epoch, BUCKET_PATH, run_name=run_name)

    test_auroc = evaluate(args.limit_test_batches, pipeline, test_dl, "test")
    results.test_auroc = test_auroc
    if is_rank_zero:
        log_metrics_gcp({'test_auroc': test_auroc})
        if last_ckpt_uri:
            try:
                from google.cloud import aiplatform
                aiplatform.log_params({"final_model_uri": last_ckpt_uri})
            except Exception as e:
                print(f"[warn] log final_model_uri failed: {e}")
    return results


# ----------------------------------------------------------------------------
# Main
# ----------------------------------------------------------------------------
def main(args: Args):
    torch.jit._state.disable()
    global_rank = int(os.environ["RANK"])
    local_rank = int(os.environ["LOCAL_RANK"])
    device = torch.device(f"cuda:{local_rank}")
    torch.cuda.set_device(device)

    if not dist.is_initialized():
        dist.init_process_group(backend="nccl")

    # 清理工作在 notebook 启动前完成；脚本内只做一次 barrier 保证同时进入数据加载
    print(f"[rank {global_rank}] process group ready", flush=True)
    dist.barrier()
    print(f"[rank {global_rank}] passed barrier, entering dataloader", flush=True)

    # 同一节点上所有 rank 共用同一个 local 目录：local leader (rank 0) 负责下载，其余 rank 等待并共享读取
    train_dl = get_dataloader_with_mosaic(INPUT_DIR_TRAIN, f"{LOCAL_CACHE_ROOT}/train", args.batch_size, "train")
    val_dl = get_dataloader_with_mosaic(INPUT_DIR_VALIDATION, f"{LOCAL_CACHE_ROOT}/val", args.batch_size, "val")
    test_dl = get_dataloader_with_mosaic(INPUT_DIR_TEST, f"{LOCAL_CACHE_ROOT}/test", args.batch_size, "test")

    # 数据加载完成后再启动 Vertex AI run（避免 rank 0 在 GCP 调用上阻塞导致其他 rank 等不到 index.json）
    run_name = f"run-{datetime.now().strftime('%Y%m%d-%H%M%S')}"
    if global_rank == 0:
        try:
            from google.cloud import aiplatform
            aiplatform.init(project=PROJECT_ID, location=REGION, experiment=EXPERIMENT_NAME)
            aiplatform.start_run(run=run_name)
            aiplatform.log_params(to_vertex_params(get_relevant_fields(args)))
        except Exception as e:
            print(f"[warn] vertex init failed: {e}")

    eb_configs = [
        EmbeddingBagConfig(
            name=f"t_{feature_name}",
            embedding_dim=args.embedding_dim,
            num_embeddings=EMB_COUNTS[idx],
            feature_names=[feature_name],
        )
        for idx, feature_name in enumerate(CAT_COLS)
    ]
    dlrm_model = DLRM(
        embedding_bag_collection=EmbeddingBagCollection(tables=eb_configs, device=torch.device("meta")),
        dense_in_features=len(DENSE_COLS),
        dense_arch_layer_sizes=args.dense_arch_layer_sizes,
        over_arch_layer_sizes=args.over_arch_layer_sizes,
        dense_device=device,
    )
    train_model = DLRMTrain(dlrm_model)

    planner = EmbeddingShardingPlanner(
        topology=Topology(
            local_world_size=get_local_size(),
            world_size=dist.get_world_size(),
            compute_device=device.type,
        ),
        batch_size=args.batch_size,
        storage_reservation=HeuristicalStorageReservation(percentage=0.05),
    )
    plan = planner.collective_plan(train_model, get_default_sharders(), dist.GroupMember.WORLD)
    if global_rank == 0 and args.print_sharding_plan:
        print(plan)
    model = DistributedModelParallel(module=train_model, device=device, plan=plan)

    dense_optimizer = KeyedOptimizerWrapper(
        dict(in_backward_optimizer_filter(model.named_parameters())),
        lambda params: torch.optim.Adagrad(params, lr=args.learning_rate, eps=args.eps),
    )
    optimizer = CombinedOptimizer([model.fused_optimizer, dense_optimizer])
    lr_scheduler = LRPolicyScheduler(optimizer, args.lr_warmup_steps, args.lr_decay_start, args.lr_decay_steps)

    train_val_test(args, model, optimizer, device, train_dl, val_dl, test_dl, lr_scheduler, run_name=run_name)

    if global_rank == 0:
        try:
            from google.cloud import aiplatform
            aiplatform.end_run()
        except Exception:
            pass

    dist.barrier()
    dist.destroy_process_group()


if __name__ == "__main__":
    main(Args())


In [ ]:
import shutil
from streaming.base.util import clean_stale_shared_memory

clean_stale_shared_memory()
shutil.rmtree("/tmp/cache", ignore_errors=True)

import random
port = random.randint(30000, 40000)
!PYTHONUNBUFFERED=1 python -m torch.distributed.launch  \
      --nproc_per_node=4 --use_env --master_port={port} \
      {BUILD_DIR}/train_dlrm_multigpu.py

# Step 6. Multi-Node Multi-GPU Training on Vertex AI Custom Training (Serverless)

This step packages the training code into a container and runs it across **multiple machines × multiple GPUs** on Vertex AI Custom Training. Unlike Step 4/5 which run inside this notebook, Step 6 submits a **remote job** — the notebook only builds the image and calls the SDK.

**Architecture**
```
Vertex AI CustomJob
├── workerpool0  (chief,   1 replica × 4 GPUs)   ← torchrun node_rank=0, hosts the rendezvous store
└── workerpool1  (workers, N replicas × 4 GPUs)  ← torchrun node_rank=1..N
```

Vertex AI injects a `CLUSTER_SPEC` env var into every container describing all node hostnames. A small `launcher.py` parses it, derives `nnodes / node_rank / master_addr`, then calls `torchrun` to spawn one process per local GPU. Each process runs `train_dlrm_multigpu.py` (the same script used in Step 5).

**Procedure**
1. 6.1 — Write `train_dlrm_multigpu.py` (self-contained training script)
2. 6.2 — Write `launcher.py` (CLUSTER_SPEC → torchrun)
3. 6.3 — Write `Dockerfile` (Python 3.10 + cu118 stack)
4. 6.4 — Build & push image to Artifact Registry
5. 6.5 — Submit the CustomJob via Vertex AI SDK
6. 6.6 — Monitor logs & view results in Vertex AI Experiments

In [ ]:
import os

PROJECT_ID       = "project-kangwe-poc"
REGION           = "us-central1"
AR_REPO          = "dlrm"                       # Artifact Registry repo name
IMAGE_URI        = f"{REGION}-docker.pkg.dev/{PROJECT_ID}/{AR_REPO}/dlrm-train:latest"
STAGING_BUCKET   = "gs://project-kangwe-poc-dlrm/staging"

BUILD_DIR        = "vertex_build"               # local dir to hold Dockerfile + scripts
os.makedirs(BUILD_DIR, exist_ok=True)
print(f"IMAGE_URI = {IMAGE_URI}")
print(f"BUILD_DIR = {os.path.abspath(BUILD_DIR)}")

## 6.1 Training script: `train_dlrm_multigpu.py`

Self-contained version of Steps 2–3 in this notebook. Key differences vs the in-notebook code:
- Reads `RANK` / `LOCAL_RANK` / `WORLD_SIZE` from env (set by torchrun) — works for 1 GPU, 1-node×N GPU, and M-node×N GPU without code changes
- All ranks on the **same node** share one `/tmp/cache` — the local leader (`LOCAL_RANK==0`) downloads shards, others read
- Only **global rank 0** talks to Vertex AI Experiments (`log_params` / `log_time_series_metrics` / `Artifact.create`)
- Checkpoints are gathered from ShardedTensors → saved to `gs://.../models/{run_name}/checkpoint_epoch_{N}.pt` → registered as `system.Model` artifacts

In [ ]:
%%writefile {BUILD_DIR}/train_dlrm_multigpu.py
#!/usr/bin/env python
"""DLRM multi-GPU / multi-node training (TorchRec + Mosaic StreamingDataset)."""
import os
import sys
import itertools
from dataclasses import dataclass, field
from functools import partial
from typing import List, Optional
from datetime import datetime

import torch
import torch.distributed as dist
from torch.optim.lr_scheduler import _LRScheduler
from torch.distributed._sharded_tensor import ShardedTensor

import torchmetrics as metrics
from tqdm import tqdm

from streaming import StreamingDataset, StreamingDataLoader

from torchrec import EmbeddingBagCollection
from torchrec.distributed import TrainPipelineSparseDist
from torchrec.distributed.comm import get_local_size
from torchrec.distributed.model_parallel import DistributedModelParallel, get_default_sharders
from torchrec.distributed.planner import EmbeddingShardingPlanner, Topology
from torchrec.distributed.planner.storage_reservations import HeuristicalStorageReservation
from torchrec.models.dlrm import DLRM, DLRMTrain
from torchrec.modules.embedding_configs import EmbeddingBagConfig
from torchrec.optim.keyed import CombinedOptimizer, KeyedOptimizerWrapper
from torchrec.optim.optimizers import in_backward_optimizer_filter
from torchrec.datasets.utils import Batch
from torchrec.sparse.jagged_tensor import KeyedJaggedTensor


# ---- dataset schema & paths (from Step 1.2 output) ----
CAT_DATA = {'cat_1': 103, 'cat_2': 180, 'cat_3': 93, 'cat_4': 15, 'cat_5': 107,
            'cat_6': 72, 'cat_7': 189, 'cat_8': 21, 'cat_9': 103, 'cat_10': 122}
DENSE_COLS = [f"int_{i}" for i in range(1, 11)]
CAT_COLS = [f"cat_{i}" for i in range(1, 11)]
EMB_COUNTS = [CAT_DATA[k] for k in CAT_COLS]

INPUT_DIR_TRAIN = "gs://project-kangwe-poc-dlrm/mds_data/train"
INPUT_DIR_VALIDATION = "gs://project-kangwe-poc-dlrm/mds_data/validation"
INPUT_DIR_TEST = "gs://project-kangwe-poc-dlrm/mds_data/test"
BUCKET_PATH = "gs://project-kangwe-poc-dlrm"
LOCAL_CACHE_ROOT = "/tmp/cache"

PROJECT_ID = "project-kangwe-poc"
REGION = "us-central1"
EXPERIMENT_NAME = "dlrm-training-demo"


@dataclass
class Args:
    epochs: int = 3
    embedding_dim: int = 128
    dense_arch_layer_sizes: list = field(default_factory=lambda: [512, 256, 128])
    over_arch_layer_sizes: list = field(default_factory=lambda: [512, 512, 256, 1])
    learning_rate: float = 0.03
    eps: float = 1e-8
    batch_size: int = 512
    print_sharding_plan: bool = True
    print_lr: bool = False
    lr_warmup_steps: int = 0
    lr_decay_start: int = 0
    lr_decay_steps: int = 0
    validation_freq: Optional[int] = None
    limit_train_batches: Optional[int] = None
    limit_val_batches: Optional[int] = None
    limit_test_batches: Optional[int] = None


@dataclass
class TrainValTestResults:
    val_aurocs: List[float] = field(default_factory=list)
    test_auroc: Optional[float] = None


def get_relevant_fields(args):
    keys = ["epochs", "embedding_dim", "dense_arch_layer_sizes",
            "over_arch_layer_sizes", "learning_rate", "eps", "batch_size"]
    result = {k: getattr(args, k) for k in keys}
    result["dense_cols"] = DENSE_COLS
    result["cat_cols"] = CAT_COLS
    result["emb_counts"] = EMB_COUNTS
    return result


def to_vertex_params(d):
    return {k: (v if isinstance(v, (int, float, bool, str)) else str(v)) for k, v in d.items()}


def transform_to_torchrec_batch(batch, num_embeddings_per_feature):
    cat_list = [batch[c].clone().detach().to(torch.float32).unsqueeze(0).T for c in DENSE_COLS]
    dense_features = torch.cat(cat_list, dim=1)
    kjt_values, kjt_lengths = [], []
    for col_idx, col_name in enumerate(CAT_COLS):
        for value in batch[col_name]:
            if value:
                kjt_values.append(int(value) % num_embeddings_per_feature[col_idx])
                kjt_lengths.append(1)
            else:
                kjt_lengths.append(0)
    sparse_features = KeyedJaggedTensor.from_lengths_sync(
        CAT_COLS, torch.tensor(kjt_values), torch.tensor(kjt_lengths, dtype=torch.int32))
    labels = batch["label"].clone().detach().to(torch.int32)
    return Batch(dense_features=dense_features, sparse_features=sparse_features, labels=labels)


transform_partial = partial(transform_to_torchrec_batch, num_embeddings_per_feature=EMB_COUNTS)


def get_dataloader_with_mosaic(remote_path, local_cache_path, batch_size, label):
    print(f"[rank {os.environ.get('RANK', '?')}] Initializing {label} from {remote_path}", flush=True)
    ds = StreamingDataset(remote=remote_path, local=local_cache_path, shuffle=True,
                          batch_size=batch_size, download_timeout=300)
    return StreamingDataLoader(ds, batch_size=batch_size)


class LRPolicyScheduler(_LRScheduler):
    def __init__(self, optimizer, num_warmup_steps, decay_start_step, num_decay_steps):
        self.num_warmup_steps = num_warmup_steps
        self.decay_start_step = decay_start_step
        self.decay_end_step = decay_start_step + num_decay_steps
        self.num_decay_steps = num_decay_steps
        if self.decay_start_step < self.num_warmup_steps:
            sys.exit("Learning rate warmup must finish before the decay starts")
        super().__init__(optimizer)

    def get_lr(self):
        step_count = self._step_count
        if step_count < self.num_warmup_steps:
            scale = 1.0 - (self.num_warmup_steps - step_count) / self.num_warmup_steps
            lr = [b * scale for b in self.base_lrs]; self.last_lr = lr
        elif self.decay_start_step <= step_count < self.decay_end_step:
            decayed = step_count - self.decay_start_step
            scale = ((self.num_decay_steps - decayed) / self.num_decay_steps) ** 2
            lr = [max(1e-7, b * scale) for b in self.base_lrs]; self.last_lr = lr
        else:
            lr = self.last_lr if self.num_decay_steps > 0 else self.base_lrs
        return lr


def gather_and_get_state_dict(model):
    rank = dist.get_rank()
    gathered = {}
    for fqn, tensor in model.state_dict().items():
        if isinstance(tensor, ShardedTensor):
            full = torch.zeros(tensor.size()).to(tensor.device) if rank == 0 else None
            tensor.gather(0, full)
            if rank == 0:
                gathered[fqn] = full
        elif rank == 0:
            gathered[fqn] = tensor
    return gathered


def log_state_dict_to_gcs(model, epoch, bucket_path, run_name=None):
    state_dict = gather_and_get_state_dict(model)
    if dist.get_rank() != 0 or not state_dict:
        return None
    from google.cloud import storage, aiplatform
    local_path = f"/tmp/checkpoint_epoch_{epoch}.pt"
    torch.save(state_dict, local_path)
    bucket_name = bucket_path.split("/")[2]
    prefix = "/".join(bucket_path.split("/")[3:]).strip("/")
    subdir = f"models/{run_name}" if run_name else "models"
    blob_path = "/".join(p for p in [prefix, subdir, f"checkpoint_epoch_{epoch}.pt"] if p)
    storage.Client().bucket(bucket_name).blob(blob_path).upload_from_filename(local_path)
    os.remove(local_path)
    gcs_uri = f"gs://{bucket_name}/{blob_path}"
    print(f"Model saved to {gcs_uri}")
    try:
        aiplatform.Artifact.create(
            schema_title="system.Model",
            display_name=f"dlrm-ckpt-epoch-{epoch}" + (f"-{run_name}" if run_name else ""),
            uri=gcs_uri, metadata={"epoch": epoch, "framework": "pytorch-torchrec"})
        aiplatform.log_params({f"checkpoint_epoch_{epoch}_uri": gcs_uri})
    except Exception as e:
        print(f"[warn] register model artifact failed: {e}")
    return gcs_uri


def log_metrics_gcp(metrics_dict, step=None):
    try:
        from google.cloud import aiplatform
        if step is not None:
            aiplatform.log_time_series_metrics(metrics_dict, step=step)
        else:
            aiplatform.log_metrics(metrics_dict)
    except Exception as e:
        print(f"[warn] vertex log failed: {e}")


def batched(it, n):
    assert n >= 1
    for x in it:
        yield itertools.chain((x,), itertools.islice(it, n - 1))


def train_one_epoch(pipeline, train_dl, val_dl, epoch, lr_scheduler, args):
    pipeline._model.train()
    iterator = itertools.islice(iter(train_dl), args.limit_train_batches)
    is_rank_zero = dist.get_rank() == 0
    pbar = tqdm(iter(int, 1), desc=f"Epoch {epoch}", total=len(train_dl), disable=not is_rank_zero)
    start_it = 0
    n = args.validation_freq if args.validation_freq else len(train_dl)
    for chunk in batched(iterator, n):
        for it in itertools.count(start_it):
            try:
                pipeline.progress(map(transform_partial, chunk))
                lr_scheduler.step()
                pbar.update(1)
            except StopIteration:
                if is_rank_zero:
                    print(f"Total number of iterations: {it}")
                start_it = it
                break
        if args.validation_freq and start_it % args.validation_freq == 0:
            evaluate(args.limit_val_batches, pipeline, val_dl, "val")
            pipeline._model.train()
    pbar.close()


def evaluate(limit_batches, pipeline, eval_dl, stage):
    pipeline._model.eval()
    device = pipeline._device
    iterator = itertools.islice(iter(eval_dl), limit_batches)
    auroc = metrics.AUROC(task="binary").to(device)
    is_rank_zero = dist.get_rank() == 0
    pbar = tqdm(iter(int, 1), desc=f"Evaluating {stage}", total=len(eval_dl), disable=not is_rank_zero)
    with torch.no_grad():
        while True:
            try:
                _loss, logits, labels = pipeline.progress(map(transform_partial, iterator))
                auroc(torch.sigmoid(logits), labels)
                pbar.update(1)
            except StopIteration:
                break
    pbar.close()
    auroc_result = auroc.compute().item()
    num_samples = torch.tensor(sum(map(len, auroc.target)), device=device)
    dist.reduce(num_samples, 0, op=dist.ReduceOp.SUM)
    if is_rank_zero:
        print(f"AUROC over {stage} set: {auroc_result}. Samples: {num_samples}")
    return auroc_result


def train_val_test(args, model, optimizer, device, train_dl, val_dl, test_dl, lr_scheduler, run_name=None):
    results = TrainValTestResults()
    pipeline = TrainPipelineSparseDist(model, optimizer, device, execute_all_batches=True)
    is_rank_zero = dist.get_rank() == 0

    val_auroc = evaluate(args.limit_val_batches, pipeline, val_dl, "val")
    results.val_aurocs.append(val_auroc)
    if is_rank_zero:
        log_metrics_gcp({'val_auroc': val_auroc}, step=0)

    last_ckpt_uri = None
    for epoch in range(args.epochs):
        train_one_epoch(pipeline, train_dl, val_dl, epoch, lr_scheduler, args)
        val_auroc = evaluate(args.limit_val_batches, pipeline, val_dl, "val")
        results.val_aurocs.append(val_auroc)
        if is_rank_zero:
            log_metrics_gcp({'val_auroc': val_auroc}, step=epoch + 1)
        last_ckpt_uri = log_state_dict_to_gcs(pipeline._model.module, epoch, BUCKET_PATH, run_name=run_name)

    test_auroc = evaluate(args.limit_test_batches, pipeline, test_dl, "test")
    results.test_auroc = test_auroc
    if is_rank_zero:
        log_metrics_gcp({'test_auroc': test_auroc})
        if last_ckpt_uri:
            try:
                from google.cloud import aiplatform
                aiplatform.log_params({"final_model_uri": last_ckpt_uri})
            except Exception as e:
                print(f"[warn] log final_model_uri failed: {e}")
    return results


def main(args: Args):
    torch.jit._state.disable()
    global_rank = int(os.environ["RANK"])
    local_rank = int(os.environ["LOCAL_RANK"])
    device = torch.device(f"cuda:{local_rank}")
    torch.cuda.set_device(device)

    if not dist.is_initialized():
        dist.init_process_group(backend="nccl")

    print(f"[rank {global_rank}] process group ready", flush=True)
    dist.barrier()
    print(f"[rank {global_rank}] passed barrier, entering dataloader", flush=True)

    train_dl = get_dataloader_with_mosaic(INPUT_DIR_TRAIN, f"{LOCAL_CACHE_ROOT}/train", args.batch_size, "train")
    val_dl = get_dataloader_with_mosaic(INPUT_DIR_VALIDATION, f"{LOCAL_CACHE_ROOT}/val", args.batch_size, "val")
    test_dl = get_dataloader_with_mosaic(INPUT_DIR_TEST, f"{LOCAL_CACHE_ROOT}/test", args.batch_size, "test")

    run_name = f"run-{datetime.now().strftime('%Y%m%d-%H%M%S')}"
    if global_rank == 0:
        try:
            from google.cloud import aiplatform
            aiplatform.init(project=PROJECT_ID, location=REGION, experiment=EXPERIMENT_NAME)
            aiplatform.start_run(run=run_name)
            aiplatform.log_params(to_vertex_params(get_relevant_fields(args)))
        except Exception as e:
            print(f"[warn] vertex init failed: {e}")

    eb_configs = [
        EmbeddingBagConfig(name=f"t_{n}", embedding_dim=args.embedding_dim,
                           num_embeddings=EMB_COUNTS[i], feature_names=[n])
        for i, n in enumerate(CAT_COLS)
    ]
    dlrm_model = DLRM(
        embedding_bag_collection=EmbeddingBagCollection(tables=eb_configs, device=torch.device("meta")),
        dense_in_features=len(DENSE_COLS),
        dense_arch_layer_sizes=args.dense_arch_layer_sizes,
        over_arch_layer_sizes=args.over_arch_layer_sizes,
        dense_device=device,
    )
    train_model = DLRMTrain(dlrm_model)

    planner = EmbeddingShardingPlanner(
        topology=Topology(local_world_size=get_local_size(), world_size=dist.get_world_size(),
                          compute_device=device.type),
        batch_size=args.batch_size,
        storage_reservation=HeuristicalStorageReservation(percentage=0.05),
    )
    plan = planner.collective_plan(train_model, get_default_sharders(), dist.GroupMember.WORLD)
    if global_rank == 0 and args.print_sharding_plan:
        print(plan)
    model = DistributedModelParallel(module=train_model, device=device, plan=plan)

    dense_optimizer = KeyedOptimizerWrapper(
        dict(in_backward_optimizer_filter(model.named_parameters())),
        lambda params: torch.optim.Adagrad(params, lr=args.learning_rate, eps=args.eps),
    )
    optimizer = CombinedOptimizer([model.fused_optimizer, dense_optimizer])
    lr_scheduler = LRPolicyScheduler(optimizer, args.lr_warmup_steps, args.lr_decay_start, args.lr_decay_steps)

    train_val_test(args, model, optimizer, device, train_dl, val_dl, test_dl, lr_scheduler, run_name=run_name)

    if global_rank == 0:
        try:
            from google.cloud import aiplatform
            aiplatform.end_run()
        except Exception:
            pass

    dist.barrier()
    dist.destroy_process_group()


if __name__ == "__main__":
    main(Args())

## 6.2 Launcher: `launcher.py`

Vertex AI does **not** set torch's `RANK`/`WORLD_SIZE` per-GPU — it only tells each **container** who it is via `CLUSTER_SPEC`. This launcher runs once per container, translates `CLUSTER_SPEC` into torchrun arguments, then execs `torchrun --nnodes=M --node_rank=K --nproc_per_node=<gpus> train_dlrm_multigpu.py`. torchrun in turn sets the per-process env vars the training script expects.

Notes:
- We use a **separate port** (`23456`) for torchrun rendezvous, not the `2222` from `CLUSTER_SPEC` (Vertex uses that for its own NCCL health check).
- On the chief node, `CLUSTER_SPEC` may list itself as `localhost` — replace with the resolvable FQDN so workers can connect.
- The launcher also does the pre-run cleanup (`clean_stale_shared_memory` + wipe `/tmp/cache`) that in Step 5 we did manually in the notebook.

In [ ]:
%%writefile {BUILD_DIR}/launcher.py
#!/usr/bin/env python
"""Vertex AI Custom Training launcher: parse CLUSTER_SPEC -> torchrun multi-node."""
import json
import os
import socket
import sys
import shutil
import subprocess

from streaming.base.util import clean_stale_shared_memory

TORCHRUN_PORT = 23456  # separate from CLUSTER_SPEC's port (2222) which Vertex uses for its own health check


def parse_cluster_spec():
    spec = os.environ.get("CLUSTER_SPEC")
    if not spec:
        return {"master_addr": "127.0.0.1", "master_port": TORCHRUN_PORT, "nnodes": 1, "node_rank": 0}

    spec = json.loads(spec)
    cluster = spec["cluster"]
    task = spec["task"]

    pools = sorted(cluster.keys())
    hosts = [h for p in pools for h in cluster[p]]
    nnodes = len(hosts)

    my_pool, my_index = task["type"], int(task["index"])
    node_rank = 0
    for p in pools:
        if p == my_pool:
            node_rank += my_index
            break
        node_rank += len(cluster[p])

    master_host = hosts[0].split(":")[0]
    if master_host in ("localhost", "127.0.0.1") and node_rank == 0:
        master_host = socket.getfqdn()
    return {"master_addr": master_host, "master_port": TORCHRUN_PORT, "nnodes": nnodes, "node_rank": node_rank}


def main():
    cfg = parse_cluster_spec()
    nproc = os.environ.get("NPROC_PER_NODE")
    if not nproc:
        import torch
        nproc = torch.cuda.device_count()

    clean_stale_shared_memory()
    shutil.rmtree("/tmp/cache", ignore_errors=True)

    cmd = [
        "torchrun",
        f"--nnodes={cfg['nnodes']}",
        f"--node_rank={cfg['node_rank']}",
        f"--nproc_per_node={nproc}",
        f"--master_addr={cfg['master_addr']}",
        f"--master_port={cfg['master_port']}",
        "train_dlrm_multigpu.py",
    ] + sys.argv[1:]

    print(f"[launcher] CLUSTER_SPEC = {os.environ.get('CLUSTER_SPEC')}", flush=True)
    print(f"[launcher] node_rank={cfg['node_rank']}/{cfg['nnodes']} "
          f"master={cfg['master_addr']}:{cfg['master_port']} nproc={nproc}", flush=True)
    print(f"[launcher] exec: {' '.join(cmd)}", flush=True)
    os.environ["PYTHONUNBUFFERED"] = "1"
    sys.exit(subprocess.call(cmd))


if __name__ == "__main__":
    main()

## 6.3 Container image: `Dockerfile`

Two version-pinning decisions matter here:
- **Python 3.10** (not 3.12): torchrun 2.2.x segfaults in the c10d rendezvous on py3.12; 3.10 is the safe choice for this torch/torchrec stack.
- **`torchrec` + `fbgemm-gpu` from the PyTorch cu118 index** (same `--index-url` as torch): the default PyPI wheels of `fbgemm-gpu==0.6.0` link against `libcudart.so.12`, which is absent on a CUDA 11.8 base image and causes `'_OpNamespace' 'fbgemm' object has no attribute 'jagged_2d_to_dense'` at import time.

`transformers` is intentionally **not** installed — it is only an optional dependency of `torchmetrics` (for text metrics) and pulling it in caused the import errors seen in Step 2.1.

In [ ]:
%%writefile {BUILD_DIR}/Dockerfile
FROM nvidia/cuda:11.8.0-cudnn8-runtime-ubuntu22.04

ENV DEBIAN_FRONTEND=noninteractive \
    PYTHONUNBUFFERED=1 \
    PIP_NO_CACHE_DIR=1

RUN apt-get update && apt-get install -y --no-install-recommends \
        python3.10 python3.10-dev python3-pip git ca-certificates && \
    ln -sf /usr/bin/python3.10 /usr/bin/python && \
    ln -sf /usr/bin/python3.10 /usr/bin/python3 && \
    rm -rf /var/lib/apt/lists/*

RUN pip install --upgrade pip && \
    pip install "numpy<2.0.0" && \
    pip install \
        torch==2.2.2 torchvision==0.17.2 torchaudio==2.2.2 \
        torchrec==0.6.0 fbgemm-gpu==0.6.0 \
        --index-url https://download.pytorch.org/whl/cu118 && \
    pip install \
        torchmetrics==1.0.3 \
        iopath==0.1.10 \
        pyre_extensions==0.0.32 \
        mosaicml-streaming==0.7.5 \
        google-cloud-aiplatform \
        google-cloud-storage \
        tqdm

WORKDIR /app
COPY train_dlrm_multigpu.py launcher.py /app/

ENTRYPOINT ["python", "launcher.py"]

## 6.4 Build & push the image to Artifact Registry

Two options — pick one:

- **Option A (recommended in a notebook): Cloud Build.** No local Docker daemon needed; the build happens on GCP and pushes directly to Artifact Registry. Takes ~10–15 min the first time (large PyTorch wheels).
- **Option B: local `docker build` + `docker push`.** Use this if the notebook host has Docker and you want to iterate locally.

Prerequisites (one-time):
- Artifact Registry API and Cloud Build API enabled on the project
- The caller has `roles/artifactregistry.writer` and `roles/cloudbuild.builds.editor`

In [ ]:
# One-time: create the Artifact Registry repo (safe to re-run, ignores "already exists")
!gcloud artifacts repositories create {AR_REPO} \
    --repository-format=docker --location={REGION} --project={PROJECT_ID} 2>/dev/null || \
    echo "repo {AR_REPO} already exists"

# ---- Option A: Cloud Build (no local docker needed) ----
!gcloud builds submit {BUILD_DIR} \
    --tag={IMAGE_URI} \
    --project={PROJECT_ID} \
    --timeout=3600s \
    --machine-type=e2-highcpu-8

# ---- Option B: local docker (uncomment if you have a docker daemon here) ----
# !gcloud auth configure-docker {REGION}-docker.pkg.dev --quiet
# !docker build -t {IMAGE_URI} {BUILD_DIR}
# !docker push {IMAGE_URI}

# Sanity check: verify the pushed image links against libcudart.so.11 (not .12)
# This runs a container remotely via Cloud Build so it works even without local docker.
# If you have local docker + GPU: docker run --rm --gpus all --entrypoint python {IMAGE_URI} -c "import torchrec; print('ok')"
print(f"\nImage ready: {IMAGE_URI}")

## 6.5 Submit the multi-node CustomJob

`worker_pool_specs[0]` is always the **chief** (must be `replica_count=1`); `worker_pool_specs[1]` holds the additional workers. Total nodes = `1 + NUM_WORKER_NODES`; total GPUs = `total_nodes × ACCELERATOR_COUNT`.

The job's service account needs:
- `roles/storage.objectAdmin` on `gs://project-kangwe-poc-dlrm` (read MDS data, write checkpoints)
- `roles/aiplatform.user` (write to Vertex AI Experiments / Metadata)

`job.run(sync=False)` returns immediately so the notebook stays responsive; use 6.6 to tail logs. Set `sync=True` if you prefer to block until completion.

In [ ]:
from google.cloud import aiplatform
import time

# ---- cluster shape ----
MACHINE_TYPE      = "n1-standard-16"
ACCELERATOR_TYPE  = "NVIDIA_TESLA_T4"
ACCELERATOR_COUNT = 4
NUM_WORKER_NODES  = 1          # workers besides the chief; total nodes = 1 + this
SERVICE_ACCOUNT   = None       # e.g. "vertex-trainer@project-kangwe-poc.iam.gserviceaccount.com"

aiplatform.init(project=PROJECT_ID, location=REGION, staging_bucket=STAGING_BUCKET)

pool_spec = {
    "machine_spec": {
        "machine_type": MACHINE_TYPE,
        "accelerator_type": ACCELERATOR_TYPE,
        "accelerator_count": ACCELERATOR_COUNT,
    },
    "replica_count": 1,
    "container_spec": {
        "image_uri": IMAGE_URI,
        "env": [
            {"name": "NCCL_DEBUG", "value": "INFO"},
            {"name": "NCCL_ASYNC_ERROR_HANDLING", "value": "1"},
        ],
    },
}

worker_pool_specs = [pool_spec]  # workerpool0 = chief
if NUM_WORKER_NODES > 0:
    worker_pool_specs.append({**pool_spec, "replica_count": NUM_WORKER_NODES})  # workerpool1 = workers

job = aiplatform.CustomJob(
    display_name="dlrm-multinode",
    worker_pool_specs=worker_pool_specs,
)

job.run(
    service_account=SERVICE_ACCOUNT,
    enable_web_access=True,
    sync=False,
)

# 等待资源在后台初始化完成
print("Submitting job...")
while not getattr(job, "_gca_resource", None) or not job._gca_resource.name:
    time.sleep(2)

print(f"Submitted: {job.resource_name}")
print(f"Console:   https://console.cloud.google.com/vertex-ai/training/custom-jobs?project={PROJECT_ID}")
JOB_ID = job.resource_name.split("/")[-1]
print(f"JOB_ID   = {JOB_ID}")

## 6.6 Monitor logs & inspect results

- **Logs**: all replica stdout/stderr go to Cloud Logging under `resource.labels.job_id`. The cell below tails them from the notebook.
- **Job state**: `job.state` / `job.wait()`.
- **Metrics & params**: Vertex AI console → Experiments → `dlrm-training-demo` → the newest run. `val_auroc` shows as a time-series chart; hyper-params and `checkpoint_epoch_*_uri` / `final_model_uri` are under the *Parameters* tab.
- **Model artifacts**: Vertex AI console → Metadata → Artifacts → filter `schema_title = system.Model`.

To load the trained model back for inference, use `inference.py` (Step 7 equivalent) which reads `final_model_uri` from the latest experiment run.

In [ ]:
# --- Check job state ---
print("state:", job.state)

# --- Read the last 100 log lines from Cloud Logging ---
!gcloud logging read \
    'resource.type="ml_job" AND resource.labels.job_id="{JOB_ID}"' \
    --project={PROJECT_ID} --limit=100 --order=asc \
    --format='value(timestamp,jsonPayload.message,textPayload)'

# --- Block until finished (optional) ---
# job.wait()
# print("final state:", job.state)

# --- After completion: list runs & metrics from Vertex AI Experiments ---
# import pandas as pd
# df = aiplatform.get_experiment_df(experiment="dlrm-training-demo")
# display(df.sort_values("run_name", ascending=False).head())

# Step 7. Inference
Since the DLRM Model's state_dicts are logged to Vertex AI Experiments, you can use the following code to load any of the saved state_dicts and create the associated DLRM model with it.

Note: The saving code and loading code is used for loading the entire DLRM model on one node and is useful as an example. In real world use cases, the expected model size could be significant (as the embedding tables can scale with the number of users or the number of products and items). It might be worthwhile to consider distributed inference.

# 7.1. Creating the DLRM model from saved state_dict
Note: You must update this with the correct run_id and path to the MLflow artifact.

In [ ]:
"""
Load the latest DLRM checkpoint from Vertex AI Experiments and run sample inference.

Equivalent of the original MLflow-based model loading + inference code, adapted to
the Vertex AI Experiments metadata (params + GCS checkpoint URI) produced by
train_dlrm_multigpu.py.
"""
import ast
import shutil

import torch
from google.cloud import aiplatform, storage

from streaming import StreamingDataset, StreamingDataLoader
from streaming.base.util import clean_stale_shared_memory

from torchrec import EmbeddingBagCollection
from torchrec.models.dlrm import DLRM
from torchrec.modules.embedding_configs import EmbeddingBagConfig
from torchrec.sparse.jagged_tensor import KeyedJaggedTensor


# ----------------------------------------------------------------------------
# Config
# ----------------------------------------------------------------------------
PROJECT_ID = "project-kangwe-poc"
REGION = "us-central1"
EXPERIMENT_NAME = "dlrm-training-demo"

TEST_DATA_REMOTE = "gs://project-kangwe-poc-dlrm/mds_data/test"
TEST_DATA_LOCAL = "/tmp/cache/infer_test"

aiplatform.init(project=PROJECT_ID, location=REGION, experiment=EXPERIMENT_NAME)


# ----------------------------------------------------------------------------
# Vertex AI Experiments helpers (replaces MLflow client calls)
# ----------------------------------------------------------------------------
def get_latest_run():
    """等价于 mlflow.search_runs(order_by=['start_time desc'], max_results=1)."""
    runs = aiplatform.ExperimentRun.list(experiment=EXPERIMENT_NAME)
    if not runs:
        raise RuntimeError(f"experiment {EXPERIMENT_NAME} 下没有任何 run")
    # run_name 形如 run-YYYYMMDD-HHMMSS，按名字倒序即最新
    return sorted(runs, key=lambda r: r.name, reverse=True)[0]


def get_latest_checkpoint_uri(params: dict) -> str:
    """等价于 get_latest_artifact_path：从 run params 里取最后一个 checkpoint 的 GCS URI."""
    if params.get("final_model_uri"):
        return params["final_model_uri"]
    ckpt_keys = sorted(
        k for k in params if k.startswith("checkpoint_epoch_") and k.endswith("_uri")
    )
    if not ckpt_keys:
        raise RuntimeError(
            "run params 里没有 checkpoint URI（final_model_uri / checkpoint_epoch_*_uri）"
        )
    return params[ckpt_keys[-1]]


def download_from_gcs(gcs_uri: str, local_path: str) -> str:
    bucket, blob = gcs_uri.replace("gs://", "").split("/", 1)
    storage.Client().bucket(bucket).blob(blob).download_to_filename(local_path)
    return local_path


def load_dlrm_from_vertex_run(run=None):
    """等价于 get_mlflow_model：从 Vertex AI run 的 params 重建 DLRM 并加载 state_dict."""
    device = torch.device("cuda")
    run = run or get_latest_run()
    print(f"Loading from experiment run: {run.name}")

    params = run.get_params()
    # list 类参数训练时用 str() 存的，用 ast.literal_eval 安全还原（不要用 eval）
    cat_cols = ast.literal_eval(params["cat_cols"])
    emb_counts = ast.literal_eval(params["emb_counts"])
    dense_cols = ast.literal_eval(params["dense_cols"])
    dense_arch_layer_sizes = ast.literal_eval(params["dense_arch_layer_sizes"])
    over_arch_layer_sizes = ast.literal_eval(params["over_arch_layer_sizes"])
    embedding_dim = int(params["embedding_dim"])

    ckpt_uri = get_latest_checkpoint_uri(params)
    print(f"Checkpoint: {ckpt_uri}")
    local_ckpt = download_from_gcs(ckpt_uri, "/tmp/dlrm_state_dict.pt")
    state_dict = torch.load(local_ckpt, map_location=device)
    # 训练时保存的是 DLRMTrain(...).module 的 state_dict，key 带 "model." 前缀 → 去掉
    state_dict = {
        (k[6:] if k.startswith("model.") else k): v for k, v in state_dict.items()
    }

    eb_configs = [
        EmbeddingBagConfig(
            name=f"t_{name}",
            embedding_dim=embedding_dim,
            num_embeddings=emb_counts[i],
            feature_names=[name],
        )
        for i, name in enumerate(cat_cols)
    ]
    dlrm_model = DLRM(
        embedding_bag_collection=EmbeddingBagCollection(tables=eb_configs, device=device),
        dense_in_features=len(dense_cols),
        dense_arch_layer_sizes=dense_arch_layer_sizes,
        over_arch_layer_sizes=over_arch_layer_sizes,
        dense_device=device,
    )
    dlrm_model.load_state_dict(state_dict)
    dlrm_model.to(device).eval()
    return dlrm_model, dense_cols, cat_cols, emb_counts

dlrm_model, dense_cols, cat_cols, emb_counts = load_dlrm_from_vertex_run()

# 7.2. Helper Function to Transform Dataloader to DLRM Inputs
The inputs that DLRM expects are dense_features and sparse_features, and so this section reuses aspects of the code from Section 3.4.2. The code shown here is verbose for clarity.

In [ ]:
def get_dataloader_with_mosaic(remote_path, local_cache_path, batch_size, label):
    print(f"Initializing {label} streaming dataset from: {remote_path}")
    dataset = StreamingDataset(
        remote=remote_path,
        local=local_cache_path,
        shuffle=False,
        batch_size=batch_size,
    )
    return StreamingDataLoader(dataset, batch_size=batch_size)


def transform_test(batch, dense_cols, cat_cols, emb_counts):
    cat_list = [
        batch[c].clone().detach().to(torch.float32).unsqueeze(0).T for c in dense_cols
    ]
    dense_features = torch.cat(cat_list, dim=1)

    kjt_values, kjt_lengths = [], []
    for col_idx, col_name in enumerate(cat_cols):
        for value in batch[col_name]:
            if value:
                kjt_values.append(int(value) % emb_counts[col_idx])
                kjt_lengths.append(1)
            else:
                kjt_lengths.append(0)
    sparse_features = KeyedJaggedTensor.from_lengths_sync(
        cat_cols,
        torch.tensor(kjt_values),
        torch.tensor(kjt_lengths, dtype=torch.int32),
    )
    return dense_features, sparse_features

# 7.3. Getting the Data

In [ ]:
num_batches = 5
batch_size = 1

clean_stale_shared_memory()
shutil.rmtree(TEST_DATA_LOCAL, ignore_errors=True)
test_dataloader = iter(
      get_dataloader_with_mosaic(TEST_DATA_REMOTE, TEST_DATA_LOCAL, batch_size, "test")
    )

# 7.4. Running Tests
In this example, you ran training for 3 epochs. The results were reasonable. Running a larger number of epochs would likely lead to optimal performance.

In [ ]:
device = torch.device("cuda:0")
for _ in range(num_batches):
    next_batch = next(test_dataloader)
    expected_result = int(next_batch["label"][0])
    dense_features, sparse_features = transform_test(
        next_batch, dense_cols, cat_cols, emb_counts
    )
    with torch.no_grad():
        logits = dlrm_model(
            dense_features=dense_features.to(device),
            sparse_features=sparse_features.to(device),
        )
        prob = torch.sigmoid(logits)[0][0].item()
    print(
        f"Expected: {expected_result}; Predicted: {round(prob)} (prob={prob:.4f})"
    )